# 01b — Data Split

**Purpose:** Split the raw dataset into train/test before any preprocessing.
This ordering is essential — preprocessing fitted on the test set leaks information.

Split strategies:
- **Temporal** (preferred): sort by `cohort_date`, take first N% as train.
  Mimics production reality where the model predicts on future customers.
- **Stratified random**: use when no date column is available.
  Stratify by event indicator to maintain equal event rates in train/test.

Validations after split:
1. Event rate parity (train ≈ test)
2. Duration distribution similarity (KS test)
3. No data leakage (no shared rows)
4. Sufficient events in both sets

---
**Inputs:** `data/raw/survival_data.parquet`, `data/raw/quality_flag.json`  
**Outputs:** `data/raw/train.parquet`, `data/raw/test.parquet`

In [ ]:
import sys, json
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split

from src.data_utils import load_data, save_data, load_config, DATA_RAW

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)
print('Ready.')

In [ ]:
cfg = load_config()
DURATION_COL = cfg['target']['duration_col']
EVENT_COL    = cfg['target']['event_col']
TEST_SIZE    = cfg['experiment']['test_size']          # default 0.20
SEED         = cfg['experiment']['random_state']

# Gate check
flag_path = DATA_RAW / 'quality_flag.json'
if flag_path.exists():
    flag = json.load(open(flag_path))
    print(f"Quality verdict: {flag['verdict']}")
    if not flag['approved']:
        raise RuntimeError('Dataset not approved — fix quality issues in notebook 02 first.')
else:
    print('quality_flag.json not found — proceeding without gate check.')

df = load_data('survival_data', stage='raw')
print(f'Shape: {df.shape}')

## 1. Choose Split Strategy

In [ ]:
# ── CONFIGURE SPLIT STRATEGY ──────────────────────────────────────────────��─
# 'temporal'   — sort by cohort_date, split chronologically (production-like)
# 'stratified' — random split stratified by event indicator
SPLIT_STRATEGY = 'temporal' if 'cohort_date' in df.columns else 'stratified'
# ─────────────────────────────────────────────────────────────────────────────

print(f'Using split strategy: {SPLIT_STRATEGY}')

In [ ]:
if SPLIT_STRATEGY == 'temporal':
    # Sort by cohort_date — older cohorts go to train, newer to test.
    # This reflects real deployment: we train on past, predict on future.
    df_sorted = df.sort_values('cohort_date').reset_index(drop=True)
    split_idx = int(len(df_sorted) * (1 - TEST_SIZE))
    split_date = df_sorted.loc[split_idx, 'cohort_date']

    df_train = df_sorted.iloc[:split_idx].copy()
    df_test  = df_sorted.iloc[split_idx:].copy()

    print(f'Temporal split at: {split_date.date()}')
    print(f'  Train: cohorts before {split_date.date()}')
    print(f'  Test : cohorts from   {split_date.date()}')

else:
    # Stratified random split — maintains event rate in both sets
    df_train, df_test = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=df[EVENT_COL],
    )
    df_train = df_train.reset_index(drop=True)
    df_test  = df_test.reset_index(drop=True)
    print('Stratified random split done.')

print(f'\nTrain: {len(df_train):,} rows  |  Test: {len(df_test):,} rows')

## 2. Split Validation

In [ ]:
issues = []

# ── Check 1: event rate parity ───────────────────────────────────────────────
train_event_rate = df_train[EVENT_COL].mean()
test_event_rate  = df_test[EVENT_COL].mean()
rate_diff = abs(train_event_rate - test_event_rate)

print(f'Event rate — Train: {train_event_rate:.3f}  |  Test: {test_event_rate:.3f}  |  Δ = {rate_diff:.3f}')
if rate_diff > 0.05:
    msg = f'Event rate difference > 5% ({rate_diff:.1%}) — consider stratified split'
    issues.append(msg)
    print(f'  ⚠️  {msg}')
else:
    print('  ✓ Event rates are similar')

# ── Check 2: no shared rows (no leakage) ─────────────────────────────────────
# Use a subset of numeric columns as a fingerprint
check_cols = [DURATION_COL, EVENT_COL]
train_fp = set(df_train[check_cols].apply(tuple, axis=1))
test_fp  = set(df_test[check_cols].apply(tuple, axis=1))
overlap  = train_fp & test_fp

print(f'\nShared rows (train ∩ test): {len(overlap)}')
if len(overlap) > 0:
    issues.append(f'{len(overlap)} rows appear in both train and test — potential leakage')
    print('  ⚠️  Overlapping rows detected!')
else:
    print('  ✓ No row overlap between train and test')

# ── Check 3: sufficient events in test set ───────────────────────────────────
n_events_test = df_test[EVENT_COL].sum()
min_events = 30
print(f'\nEvents in test set: {n_events_test}')
if n_events_test < min_events:
    msg = f'Only {n_events_test} events in test set (minimum recommended: {min_events})'
    issues.append(msg)
    print(f'  ⚠️  {msg}')
else:
    print(f'  ✓ Sufficient events in test set')

# ── Check 4: KS test on duration distributions ───────────────────────────────
ks_stat, ks_p = stats.ks_2samp(df_train[DURATION_COL], df_test[DURATION_COL])
print(f'\nDuration KS test — stat: {ks_stat:.4f}  p-value: {ks_p:.4f}')
if ks_p < 0.01:
    msg = f'Duration distributions differ significantly (KS p={ks_p:.4f}) — check temporal drift'
    issues.append(msg)
    print(f'  ⚠️  {msg}')
else:
    print('  ✓ Duration distributions are compatible')

# ── Final verdict ─────────────────────────────────────────────────────────────
print(f'\n{"=" * 50}')
if issues:
    print(f'Split warnings ({len(issues)}):')
    for i in issues:
        print(f'  ⚠️  {i}')
else:
    print('✓ All split validation checks passed.')
print('=' * 50)

## 3. Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Duration distributions
axes[0].hist(df_train[DURATION_COL], bins=30, alpha=0.6, color='steelblue', label='Train', density=True)
axes[0].hist(df_test[DURATION_COL],  bins=30, alpha=0.6, color='tomato',    label='Test',  density=True)
axes[0].set(title='Duration Distribution', xlabel='Months')
axes[0].legend()

# Event rate comparison
x = ['Train', 'Test']
y = [train_event_rate, test_event_rate]
bars = axes[1].bar(x, y, color=['steelblue', 'tomato'])
for bar, v in zip(bars, y):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.1%}', ha='center')
axes[1].set(title='Event Rate', ylabel='Event rate', ylim=(0, max(y) * 1.2))
axes[1].axhline(0.05, color='gray', linestyle='--', alpha=0.5, label='5% threshold')
axes[1].legend()

# Split size
axes[2].pie([len(df_train), len(df_test)],
            labels=[f'Train\n{len(df_train):,}', f'Test\n{len(df_test):,}'],
            colors=['steelblue', 'tomato'], autopct='%1.0f%%', startangle=90)
axes[2].set_title(f'Split: {1-TEST_SIZE:.0%} / {TEST_SIZE:.0%}')

plt.suptitle(f'Train/Test Split — Strategy: {SPLIT_STRATEGY}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Temporal split — cohort date timeline
if 'cohort_date' in df.columns:
    fig, ax = plt.subplots(figsize=(13, 3))
    train_monthly = df_train.set_index('cohort_date').resample('ME').size()
    test_monthly  = df_test.set_index('cohort_date').resample('ME').size()
    train_monthly.plot(ax=ax, color='steelblue', label='Train', linewidth=2)
    test_monthly.plot(ax=ax, color='tomato',    label='Test',  linewidth=2)
    ax.axvline(split_date if SPLIT_STRATEGY == 'temporal' else df['cohort_date'].quantile(0.8),
               color='black', linestyle='--', label='Split date')
    ax.set(title='Cohort registrations over time', xlabel='Date', ylabel='Count')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Split Summary Table

In [ ]:
numeric_cols = cfg.get('numeric_features', []) + [DURATION_COL]
numeric_cols = [c for c in numeric_cols if c in df.columns]

summary_table = pd.DataFrame({
    'Train': df_train[numeric_cols].mean(),
    'Test':  df_test[numeric_cols].mean(),
}).T

summary_table.loc['Difference (%)'] = (
    (summary_table.loc['Test'] - summary_table.loc['Train'])
    / summary_table.loc['Train'].replace(0, np.nan) * 100
).round(1)

print('Mean feature values: Train vs Test')
display(summary_table.round(3))

## 5. Save Splits

In [ ]:
save_data(df_train, 'train', stage='raw')
save_data(df_test,  'test',  stage='raw')

# Save split metadata
split_meta = {
    'strategy'         : SPLIT_STRATEGY,
    'n_train'          : len(df_train),
    'n_test'           : len(df_test),
    'train_event_rate' : round(train_event_rate, 4),
    'test_event_rate'  : round(test_event_rate, 4),
    'test_size'        : TEST_SIZE,
    'split_date'       : str(split_date.date()) if SPLIT_STRATEGY == 'temporal' else None,
    'warnings'         : issues,
}
import json
with open(DATA_RAW / 'split_metadata.json', 'w') as f:
    json.dump(split_meta, f, indent=2)

print('\nSplit metadata saved.')
print('\n✓ Done. Proceed to 02_data_quality.ipynb or 02b_preprocessing.ipynb')